In [1]:
import pandas as pd
import numpy as np

In [4]:
# !pip install lxml
import sys
print(sys.executable)

C:\Users\Shyam\Cryptomedbench\tf_env\Scripts\python.exe


In [18]:
!{sys.executable} -m pip install lxml
!{sys.executable} -m pip install bioc


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Shyam\Cryptomedbench\tf_env\Scripts\python.exe -m pip install --upgrade pip


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13858 sha256=094524ad26c01d957854ccdcf533e34be641ae861a009d489c7589d143105feb
  Stored in directory: c:\users\shyam\appdata\local\pip\cache\wheels\fc\ab\d4\5da2067ac95b36618c629a5f93f809425700506f72c9732fac
Successfully built docopt

   ---------------------------------------- 0/5 [sortedcontainers]
   ---------------- ----------------------- 2/5 [jsonlines]
   -------------------------------- ------- 4/5 [bioc]
   -------------------------------- -


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Shyam\Cryptomedbench\tf_env\Scripts\python.exe -m pip install --upgrade pip


In [36]:
import bioc
import pandas as pd


def load_bc5cdr(filepath):
    rows = []

    with open(filepath, 'r', encoding='utf8') as fp:
        collection = bioc.load(fp)

    for document in collection.documents:
        doc_id = document.id

        # Combine title + abstract
        full_text = ""
        mesh_to_text = {}

        for passage in document.passages:
            if passage.text:
                full_text += passage.text + " "

            # Build MESH → entity text mapping
            for annotation in passage.annotations:
                mesh_id = annotation.infons.get("MESH")
                entity_text = annotation.text
                entity_type = annotation.infons.get("type")

                if mesh_id:
                    mesh_to_text[mesh_id] = {
                        "text": entity_text,
                        "type": entity_type
                    }

        full_text = full_text.strip()

        # Extract relations (document-level)
        for relation in document.relations:

            if relation.infons.get("relation") == "CID":

                chemical_mesh = relation.infons.get("Chemical")
                disease_mesh = relation.infons.get("Disease")

                chemical_text = mesh_to_text.get(chemical_mesh, {}).get("text")
                disease_text = mesh_to_text.get(disease_mesh, {}).get("text")

                rows.append({
                    "doc_id": doc_id,
                    "text": full_text,
                    "chemical_mesh": chemical_mesh,
                    "chemical_text": chemical_text,
                    "disease_mesh": disease_mesh,
                    "disease_text": disease_text,
                    "relation_type": "CID"
                })

    return pd.DataFrame(rows)

In [46]:
train_df = load_bc5cdr("CDR.Corpus.v010516/CDR_TrainingSet.BioC.xml")
train_df['relation_type'].unique()

array(['CID'], dtype=object)

In [38]:
dev_df = load_bc5cdr("CDR.Corpus.v010516/CDR_DevelopmentSet.BioC.xml")
dev_df.head()

,doc_id,text,chemical_mesh,chemical_text,disease_mesh,disease_text,relation_type
0,6794356,Tricuspid valve regurgitation and lithium carb...,D016651,Lithium carbonate,D003490,cyanosis,CID
1,6794356,Tricuspid valve regurgitation and lithium carb...,D016651,Lithium carbonate,D001145,cardiac arrhythmia,CID
2,6794356,Tricuspid valve regurgitation and lithium carb...,D016651,Lithium carbonate,D003866,neurologic depression,CID
3,6504332,Phenobarbital-induced dyskinesia in a neurolog...,D010634,Phenobarbital,D004409,dyskinesia,CID
4,6436733,Acute changes of blood ammonia may predict sho...,D014635,VPA,D006970,drowsiness,CID


In [45]:
test_df = load_bc5cdr("CDR.Corpus.v010516/CDR_TestSet.BioC.xml")

In [40]:
print(dev_df.describe())
print(dev_df.info())

         doc_id                                               text  \
count      1012                                               1012   
unique      500                                                500   
top     8888541  Serotonin syndrome from venlafaxine-tranylcypr...   
freq         16                                                 16   

       chemical_mesh chemical_text disease_mesh disease_text relation_type  
count           1012          1012         1012         1010          1012  
unique           347           431          316          519             1  
top          D004317    amiodarone      D012640  hypotension           CID  
freq              24            15           41           25          1012  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   doc_id         1012 non-null   object
 1   text           1012 non-

In [50]:
print(train_df.info())
print(train_df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5674 entries, 0 to 5673
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   doc_id         5674 non-null   object
 1   text           5674 non-null   object
 2   chemical_text  5674 non-null   object
 3   disease_text   5674 non-null   object
 4   label          5674 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 221.8+ KB
None
             label
count  5674.000000
mean      0.182763
std       0.386507
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [47]:
def create_classification_dataset(filepath):
    import bioc
    import pandas as pd
    from itertools import product

    rows = []

    with open(filepath, 'r', encoding='utf8') as fp:
        collection = bioc.load(fp)

    for document in collection.documents:
        doc_id = document.id
        full_text = ""
        mesh_to_text = {}
        chemicals = set()
        diseases = set()
        positive_pairs = set()

        # Extract text + entities
        for passage in document.passages:
            if passage.text:
                full_text += passage.text + " "

            for annotation in passage.annotations:
                mesh = annotation.infons.get("MESH")
                ent_type = annotation.infons.get("type")
                ent_text = annotation.text

                if mesh:
                    mesh_to_text[mesh] = ent_text

                    if ent_type == "Chemical":
                        chemicals.add(mesh)
                    elif ent_type == "Disease":
                        diseases.add(mesh)

        full_text = full_text.strip()

        # Extract positive relations
        for relation in document.relations:
            if relation.infons.get("relation") == "CID":
                chem = relation.infons.get("Chemical")
                dis = relation.infons.get("Disease")
                positive_pairs.add((chem, dis))

        # Generate all possible pairs
        for chem, dis in product(chemicals, diseases):

            label = 1 if (chem, dis) in positive_pairs else 0

            rows.append({
                "doc_id": doc_id,
                "text": full_text,
                "chemical_text": mesh_to_text.get(chem),
                "disease_text": mesh_to_text.get(dis),
                "label": label
            })

    return pd.DataFrame(rows)

In [48]:
train_df = create_classification_dataset(
    "CDR.Corpus.v010516/CDR_TrainingSet.BioC.xml"
)

train_df.head()

,doc_id,text,chemical_text,disease_text,label
0,227508,Naloxone reverses the antihypertensive effect ...,alpha-methyldopa,hypotensive,1
1,227508,Naloxone reverses the antihypertensive effect ...,alpha-methyldopa,hypertensive,0
2,227508,Naloxone reverses the antihypertensive effect ...,naloxone,hypotensive,0
3,227508,Naloxone reverses the antihypertensive effect ...,naloxone,hypertensive,0
4,227508,Naloxone reverses the antihypertensive effect ...,clonidine,hypotensive,0


In [49]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5674 entries, 0 to 5673
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   doc_id         5674 non-null   object
 1   text           5674 non-null   object
 2   chemical_text  5674 non-null   object
 3   disease_text   5674 non-null   object
 4   label          5674 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 221.8+ KB
